# Enhanced Combined Drought Index - Precipitation Drought Index

* **Products used:** 
[rainfall_chirps_daily](https://explorer.digitalearth.africa/products/rainfall_chirps_daily)

## Background 

Drought is an extended period, during which, fresh water availability and accessibility for the ecosystem at a given time and place is below normal, due to unfavourable spatial and temporal distribution of rainfall, temperature, soil moisture and wind characteristics [(Balint et al., 2013)](https://doi.org/10.1016/B978-0-444-59559-1.00023-2). Severe droughts can affect large populations, leading to a long-term threat to people’s livelihoods and result in tremendous economic loss [(Enenkel et al., 2016)](https://doi.org/10.3390/rs8040340).

The Enhanced Combined Drought Index aims at the timely and reliable detection of drought events with regard to their spatio-temporal extent and severity. The Enhanced Drought Index is a combination of the following: a precipitation component, which considers rainfall deficits and dryness persistence; a soil moisture component, which considers soil moisture deficits and deficit persistence; a vegetation component which considers NDVI deficits and deficit persistence; and a temperature component, which considers temperature excesses and persistence of high temperatures. The index uses satellite-derived rainfall, soil moisture, land surface temperature and vegetation status as input datasets. [(Enenkel et al., 2016)](https://doi.org/10.3390/rs8040340)

The **Enhanced Drought Index** is a combination of the following components: Vegetation, Precipitation, Temperature and Soil moisture.

This notebook will focus on the **Precipitation component**, component, which considers rainfall deficits and dryness persistence. The index calculated using the precipitation component is referred as the Precipitation Drought Index (PDI)." The Precipitation Drought Index (PDI) is a commonly used indicator for assessing drought conditions based on precipitation deficits. It measures the difference between observed precipitation and expected or normal precipitation over a specific time period, typically in relation to the long-term average.

The notebook outlines:

***
1. Loading Python and DE Africa Packages
2. Setting dask cluster
3. Loading area on interest
5. Load Landsat Surface reflectance and Calculate the Normalized Difference Vegetation Index (NDVI)
6. Resampling of data to dekadal (10-day) intervals
7. Calculate the Vegetation Drought Index
8. Export the result as zip

## Getting started

To run this analysis, run all the cells in the notebook, starting with the "Load packages" cell. 

### Load packages

In [1]:
# import calendar
import json
import os
# from datetime import datetime, timedelta

import dask
import geopandas as gpd
import numpy as np
import pandas as pd
import rioxarray
import toolz
import xarray as xr
from datacube import Datacube
from deafrica_tools.bandindices import calculate_indices
from deafrica_tools.dask import create_local_dask_cluster
from deafrica_tools.datahandling import load_ard
from odc.geo.geobox import GeoBox
from odc.geo.geom import Geometry
from odc.geo.xr import rasterize
from pyarrow import Table
from pyarrow.parquet import write_table
# from xarray.core.groupby import DataArrayGroupBy, DatasetGroupBy

### Set up a Dask cluster

Dask can be used to better manage memory use and conduct the analysis in parallel. 
For an introduction to using Dask with Digital Earth Africa, see the [Dask notebook](../../Beginners_guide/08_Parallel_processing_with_dask.ipynb).

>**Note**: We recommend opening the Dask processing window to view the different computations that are being executed; to do this, see the *Dask dashboard in DE Africa* section of the [Dask notebook](../../Beginners_guide/08_Parallel_processing_with_dask.ipynb).

To use Dask, set up the local computing cluster using the cell below.

In [2]:
create_local_dask_cluster()

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/vikinyaga@gmail.com/proxy/8787/status,
Dashboard: /user/vikinyaga@gmail.com/proxy/8787/status,Workers: 1
Total threads: 15,Total memory: 97.21 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:44709,Workers: 1
Dashboard: /user/vikinyaga@gmail.com/proxy/8787/status,Total threads: 15
Started: Just now,Total memory: 97.21 GiB
Comm: tcp://127.0.0.1:45791,Total threads: 15
Dashboard: /user/vikinyaga@gmail.com/proxy/38705/status,Memory: 97.21 GiB
Nanny: tcp://127.0.0.1:37829,


### Analysis parameters

The following cell sets important parameters for the analysis:

* `country`: The country to run the analysis over.
* `resolution`: The x and y cell resolution of the satellite data in metres (spatial resolution). We'll use 5,000 m, which is approximately equal to the default CHIRPS resolution.
* `output_crs` : The coordinate reference system that the loaded data is to be reprojected to.
* `dask_chunks`:  the size of the dask chunks, dask breaks data into manageable chunks that can be easily stored in memory, e.g. `dict(x=1000,y=1000)`
* `time_range` : Time range to load data for.
* `ip` : The interest period to use to calculate the drought indices e.g. (3, 4, 5 dekads). It defines to what extent past observations are considered. Longer IPs detect more severe drought events. For example, if the IP=3 dekads, then the drought index (say PDI) of 0.35 for dekad 2 of 2006 implies actual drought for dekad 36 of 2005, dekad 1 of 2006 and dekad 2 of 2006.
* `output_dir` :  The directory in which to store results from the analysis.

**If running the notebook for the first time**, keep the default settings below.
This will demonstrate how the analysis works and provide meaningful results.

In [3]:
resolution = (-5000, 5000)
output_crs = "EPSG:6933"
dask_chunks = dict(x=3200,y=3200)
time_range = ("2014","2024")
output_dir = "results"
# Corresponding to the six-months Standardized Precipitation Evapotranspiration Index
ip = 18

In [4]:
# Create the outpur directory if it does not exist.
os.makedirs(output_dir, exist_ok=True)

In [5]:
african_countries_url = "https://raw.githubusercontent.com/digitalearthafrica/deafrica-sandbox-notebooks/main/Supplementary_data/Rainfall_anomaly_CHIRPS/african_countries.geojson"
african_countries = gpd.read_file(african_countries_url)
print(np.unique(african_countries.COUNTRY))

['Algeria' 'Angola' 'Benin' 'Botswana' 'Burkina Faso' 'Burundi' 'Cameroon'
 'Cape Verde' 'Central African Republic' 'Chad' 'Comoros'
 'Congo-Brazzaville' 'Cote d`Ivoire' 'Democratic Republic of Congo'
 'Djibouti' 'Egypt' 'Equatorial Guinea' 'Eritrea' 'Ethiopia' 'Gabon'
 'Gambia' 'Ghana' 'Guinea' 'Guinea-Bissau' 'Kenya' 'Lesotho' 'Liberia'
 'Libya' 'Madagascar' 'Malawi' 'Mali' 'Mauritania' 'Morocco' 'Mozambique'
 'Namibia' 'Niger' 'Nigeria' 'Rwanda' 'Sao Tome and Principe' 'Senegal'
 'Sierra Leone' 'Somalia' 'South Africa' 'Sudan' 'Swaziland' 'Tanzania'
 'Togo' 'Tunisia' 'Uganda' 'Western Sahara' 'Zambia' 'Zimbabwe']


In [6]:
# Load the area of interest
country = "Ethiopia"

idx = african_countries[african_countries['COUNTRY'] == country].index[0]
geopolygon = Geometry(geom=african_countries.iloc[idx].geometry, crs=african_countries.crs)

In [7]:
# Create the output path
output_path = os.path.join(output_dir, f"PDI_{country}.parquet")

### Connect to the datacube

Connect to the datacube so we can access DE Africa data.
The `app` parameter is a unique name for the analysis which is based on the notebook file name.

In [8]:
# Connect to the datacube
dc = Datacube(app="PrecipitationDroughtIndex")

## Load CHIRPS data
Load the CHIRPS daily data from the datacube using the analysis parameters set in the previous section.

In [9]:
%%time
ds_rf = dc.load(product='rainfall_chirps_daily',
                measurements=["rainfall"],
                time=time_range,
                dask_chunks=dask_chunks,
                geopolygon=geopolygon,
                output_crs=output_crs,
                resolution=resolution,
               )
                    
ds_rf

CPU times: user 39.5 s, sys: 10.6 s, total: 50 s
Wall time: 38.4 s


<xarray.Dataset>
Dimensions:      (time: 3712, y: 291, x: 290)
Coordinates:
  * time         (time) datetime64[ns] 2014-01-01T11:59:59.500000 ... 2024-04...
  * y            (y) float64 1.882e+06 1.878e+06 ... 4.375e+05 4.325e+05
  * x            (x) float64 3.182e+06 3.188e+06 ... 4.622e+06 4.628e+06
    spatial_ref  int32 6933
Data variables:
    rainfall     (time, y, x) float32 dask.array<chunksize=(1, 291, 290), meta=np.ndarray>
Attributes:
    crs:           EPSG:6933
    grid_mapping:  spatial_ref

In [10]:
# Rasterize the country geopolygon and mask the loaded  rainfall data.
country_mask = rasterize(poly=geopolygon, how=ds_rf.odc.geobox)
ds_rf = ds_rf.where(country_mask)

In [11]:
# Convert to DataArray
da_rf = ds_rf["rainfall"]

# Set -9999 no-data values to NaN
da_rf = da_rf.where(da_rf !=-9999.)

da_rf

<xarray.DataArray 'rainfall' (time: 3712, y: 291, x: 290)>
dask.array<where, shape=(3712, 291, 290), dtype=float32, chunksize=(1, 291, 290), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 2014-01-01T11:59:59.500000 ... 2024-04...
  * y            (y) float64 1.882e+06 1.878e+06 ... 4.375e+05 4.325e+05
  * x            (x) float64 3.182e+06 3.188e+06 ... 4.622e+06 4.628e+06
    spatial_ref  int32 6933
Attributes:
    units:         mm
    nodata:        -9999
    crs:           EPSG:6933
    grid_mapping:  spatial_ref

## Resample data

Resample the rainfall `da_rf` timeseries into **dekadal** (10-day) timesteps.

In [12]:
# def get_dekad(date: np.datetime64) -> np.datetime64:
#     """
#     Get the start date of the dekad that a date belongs to.
#     Every month has three dekads, such that the first two dekads
#     have 10 days (i.e., 1-10, 11-20), and the third is comprised of the
#     remaining days of the month.

#     Parameters
#     ----------
#     date : np.datetime64
#         Date to check.

#     Returns
#     -------
#     np.datetime64
#         Start date of the dekad.
#     """
#     timestamp = pd.Timestamp(date)
#     year = timestamp.year
#     month = timestamp.month

#     first_day = datetime(year, month, 1)
#     last_day = datetime(year, month, calendar.monthrange(year, month)[1])

#     d1_start_date, d2_start_date, d3_start_date = pd.date_range(
#         start=first_day, end=last_day, freq="10D", inclusive="left"
#     )

#     if d1_start_date <= timestamp < d2_start_date:
#         return np.datetime64(d1_start_date, "ns")
#     elif d2_start_date <= timestamp < d3_start_date:
#         return np.datetime64(d2_start_date, "ns")
#     else:
#         return np.datetime64(d3_start_date, "ns")

# def group_by_dekad(
#     ds: xr.DataArray | xr.Dataset,
# ) -> DataArrayGroupBy | DatasetGroupBy:
#     """
#     Group a dataset or array by dekad.

#     Parameters
#     ----------
#     ds : xr.DataArray | xr.Dataset
#         Dataset or DataArray to group

#     Returns
#     -------
#     xr.core.groupby.DataArrayGroupBy | xr.core.groupby.DatasetGroupBy
#         Groupby oject
#     """
#     group = xr.DataArray(
#         data=np.vectorize(get_dekad)(ds.time.values),
#         coords=ds.time.coords,
#         dims=ds.time.dims,
#         name="dekad",
#         attrs=ds.time.attrs,
#     )
#     grouped_by_dekad = ds.groupby(group)
#     return grouped_by_dekad

In [13]:
%%time
# Resample the rainfall data from daily to decadal (10-day) intervals
da_rf_dekadal = group_by_dekad(da_rf).mean(dim="time").compute()
da_rf_dekadal

/usr/local/lib/python3.10/dist-packages/rasterio/warp.py:344: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  _reproject(


CPU times: user 16.9 s, sys: 632 ms, total: 17.5 s
Wall time: 1min 22s


<xarray.DataArray 'rainfall' (dekad: 366, y: 291, x: 290)>
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
...
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]], dtype=float32)
Coordinates:
  * y            (y) float64 1.882e+06 1.878e+06 ... 4.375e+05 4.325e+05
  * x            (x) float64 3.182e+06 3.188e+06 ... 4.622e+06 4.628e+06
    spatial_ref  int32 6933
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-04-21
Attributes:
    units:         mm
    nodata:        -9999
    crs:           EPSG:6933
    grid_mapping:  spatial_ref

## Modify the timeseries

The raw timeseries of precipitation is modified to adjust the range of all variables to avoid a division by zero.

In [14]:
da_rf_dekadal_modified =  da_rf_dekadal + 1
da_rf_dekadal_modified

<xarray.DataArray 'rainfall' (dekad: 366, y: 291, x: 290)>
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
...
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]], dtype=float32)
Coordinates:
  * y            (y) float64 1.882e+06 1.878e+06 ... 4.375e+05 4.325e+05
  * x            (x) float64 3.182e+06 3.188e+06 ... 4.622e+06 4.628e+06
    spatial_ref  int32 6933
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-04-21

## Group the modified timeseries using the `ip` parameter

The `ip` parameter determines to what extent past observations are considered. Longer IPs detect more severe drought events. The default `ip` used of **18 dekads** corresponds to the 6-month Standardized Precipitation Evapotranspiration Index.

For example, calculating the Precipitation Drought Index for the dekad `2011-01-11` using an interest period of `ip=3` requires rainfall data from the dekad `2010-12-21`, `2011-01-01` and `2011-01-11`.


**Each interest period is labelled using its end dekad.**

In [15]:
# def get_dekad_no_in_month(date: np.datetime64) -> int:
#     """
#     Get the number of the dekad in a month that a date belongs to.
#     Every month has three dekads, such that the first two dekads
#     have 10 days (i.e., 1-10, 11-20), and the third is comprised of the
#     remaining days of the month.

#     Parameters
#     ----------
#     date : np.datetime64
#         Date to check.

#     Returns
#     -------
#     int
#         Number of the dekad in a month.
#     """
#     timestamp = pd.Timestamp(date)
#     year = timestamp.year
#     month = timestamp.month

#     first_day = datetime(year, month, 1)
#     last_day = datetime(year, month, calendar.monthrange(year, month)[1])

#     d1_start_date, d2_start_date, d3_start_date = pd.date_range(
#         start=first_day, end=last_day, freq="10D", inclusive="left"
#     )

#     if d1_start_date <= timestamp < d2_start_date:
#         return 1
#     elif d2_start_date <= timestamp < d3_start_date:
#         return 2
#     else:
#         return 3


# def get_dekad_no_in_year(date: np.datetime64) -> int:
#     """
#     Get the number of the dekad in a year that a date belongs to.
#     Every month has three dekads, such that the first two dekads
#     have 10 days (i.e., 1-10, 11-20), and the third is comprised of the
#     remaining days of the month (21-last day). Every year has 36 dekads.

#     Parameters
#     ----------
#     date : np.datetime64
#         Date to check.

#     Returns
#     -------
#     int
#         Number of the dekad in a year.
#     """
#     dekad_no_in_month = get_dekad_no_in_month(date=date)
#     timestamp = pd.Timestamp(date)
#     month = timestamp.month
#     dekad_no_in_year = ((month - 1) * 3) + dekad_no_in_month
#     return dekad_no_in_year


# def get_interest_period(dekad: np.datetime64, ip: int) -> list[np.datetime64]:
#     """
#     Get all the dekads in the interest period for a dekad.
#     `dekad` will always be the end dekad of the interest period.

#     Parameters
#     ----------
#     dekad : np.datetime64
#         Dekad to get the interest period for.
#         Will always be the end dekad of the interest period
#     ip : int
#         Number of dekads in an interest period.

#     Returns
#     -------
#     list[np.datetime64]
#         All the dekads in the interest period.
#     """
#     year = pd.Timestamp(dekad).year
#     dekad_no_in_year = get_dekad_no_in_year(dekad) - (ip - 1)
#     while dekad_no_in_year <= 0:
#         year -= 1
#         dekad_no_in_year += 36

#     month = (dekad_no_in_year - 1) // 3 + 1
#     dekad_no_in_month = (dekad_no_in_year - 1) % 3 + 1
#     if dekad_no_in_month == 1:
#         day = 1
#     elif dekad_no_in_month == 2:
#         day = 11
#     elif dekad_no_in_month == 3:
#         day = 21

#     start_dekad = np.datetime64(datetime(year, month, day), "ns")
#     date_range = pd.date_range(
#         start_dekad, dekad, freq=timedelta(days=1), inclusive="both"
#     ).values
#     interest_period_dekads = np.unique(np.vectorize(get_dekad)(date_range))
#     return interest_period_dekads


# def bin_by_interest_period(
#     ds: xr.Dataset | xr.DataArray, ip: int
# ) -> dict[np.datetime64, xr.Dataset | xr.DataArray]:
#     """
#     Bin each dekad in the dataset by interest period.

#     Parameters
#     ----------
#     ds : xr.Dataset | xr.DataArray
#         Dataset to bin
#     ip : int
#         Number of dekads in an interest period.

#     Returns
#     -------
#     list[tuple[np.datetime64, np.datetime64]]
#         List of dekad ranges to bin
#     """
#     start_date = ds.dekad.min().values
#     end_date = ds.dekad.max().values
#     date_range = pd.date_range(
#         start_date, end_date, freq=timedelta(days=1), inclusive="both"
#     ).values
#     dekads = np.unique(np.vectorize(get_dekad)(date_range))
#     bins = {i: get_interest_period(dekad=i, ip=ip) for i in dekads}
#     binned_by_interest_period = {interest_period_label : ds.reindex(dekad=interest_period) for interest_period_label, interest_period in bins.items()}
#     return binned_by_interest_period

In [16]:
# Group the modified rainfall dekadal timeseries by the interest period.
# The dictionary keys are the end dekad for each interest period.
# The values of the dictionary are the rainfall data that belongs to the interest period
da_rf_binned_by_interest_period = bin_by_interest_period(ds=da_rf_dekadal_modified, ip=ip)

In [17]:
# Get the interest periods
interest_periods = list(da_rf_binned_by_interest_period.keys())

## Get the average values for each interest period

Get the actual average rainfall for each interest period.

In [18]:
da_rf_actual_avg_for_ip = xr.concat([da_rf_binned_by_interest_period[i].mean(dim="dekad").assign_coords(dekad=i).expand_dims({"dekad": 1}) for i in interest_periods], dim="dekad")
da_rf_actual_avg_for_ip

<xarray.DataArray 'rainfall' (dekad: 372, y: 291, x: 290)>
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
...
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]], dtype=float32)
Coordinates:
  * y            (y) float64 1.882e+06 1.878e+06 ... 4.375e+05 4.325e+05
  * x            (x) float64 3.182e+06 3.188e+06 ... 4.622e+06 4.628e+06
    spatial_ref  int32 6933
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-04-21

## Get the long term average for each interest period over the years of available data


In [19]:
# Group the interest periods by year by getting the dekad number in the year for the end dekad of
# the interest period. 
grouped_by_year = toolz.groupby(lambda dekad: get_dekad_no_in_year(date=dekad), interest_periods)

Get the long term average rainfall for each interest period.

In [20]:
long_term_avg_rf = {}
for dekad_no_in_year, interest_periods_list in grouped_by_year.items():
    long_term_avg_for_period = xr.concat([da_rf_binned_by_interest_period[i] for i in interest_periods_list], dim="dekad").mean(dim="dekad")
    long_term_avg_rf[dekad_no_in_year] = long_term_avg_for_period
    
da_rf_long_term_avg_for_ip = xr.concat([long_term_avg_rf[get_dekad_no_in_year(i)].assign_coords(dekad=i).expand_dims({"dekad": 1}) for  i in interest_periods], dim="dekad")

assert all(da_rf_actual_avg_for_ip.dekad.values == da_rf_long_term_avg_for_ip.dekad.values)

da_rf_long_term_avg_for_ip

<xarray.DataArray 'rainfall' (dekad: 372, y: 291, x: 290)>
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
...
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]], dtype=float32)
Coordinates:
  * y            (y) float64 1.882e+06 1.878e+06 ... 4.375e+05 4.325e+05
  * x            (x) float64 3.182e+06 3.188e+06 ... 4.622e+06 4.628e+06
    spatial_ref  int32 6933
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-04-21

## Get the actual length of continous deficit in each interest period

For rainfall the run length is the number of successive dekads in an interest period below the long term average for the same interest period.

In [21]:
# def get_no_data_mask(arr):
#     """
#     Check if all values in an array are NaN
#     """
#     return np.all(np.isnan(arr))

# # From https://www.geeksforgeeks.org/maximum-consecutive-ones-or-zeros-in-a-binary-array/
# def max_consecutive_ones(arr: np.ndarray) -> int:
#     """
#     Get the maximum number of successive ones in an array.

#     Parameters
#     ----------

#     arr : np.ndarray
#         Array to check

#     Returns
#     -------
#     int
#         Maximum number of consecutive ones in the input array.

#     """

#     n = len(arr)
#     # initialize count
#     count = 0
#     # initialize max
#     result = 0

#     for i in range(0, n):
#         # If 1 is found, increment count
#         # and update result if count
#         # becomes more.
#         if arr[i] == 1:
#             # increase count
#             count += 1
#             result = max(result, count)
#         # Reset count if one is not found
#         else:
#             count = 0

#     return result

In [22]:
%%time
# For each interest period get the maximum number of successive dekads below long term average rainfall in the interest period.
rf_run_length = []
for interest_period in interest_periods:
    # Get the dekads in the interest period
    ds = da_rf_binned_by_interest_period[interest_period]
    
    # Identify pixels which are empty for all dekads
    no_data_mask = xr.apply_ufunc(
        get_no_data_mask,
        ds,
        input_core_dims=[["dekad"]],
        vectorize=True,
        dask="allowed",
    )
    
    # Get the long term average for the interest period
    long_term_avg = da_rf_long_term_avg_for_ip.sel(dekad=interest_period)
    
    # Get the pixels where the rainfall is below the long term average rainfall
    mask = xr.where(ds < long_term_avg, 1, 0)
    
    # Get the maximum number of successive dekads below long term average rainfall.
    actual_run_length = xr.apply_ufunc(
        max_consecutive_ones,
        mask,
        input_core_dims=[["dekad"]],
        vectorize=True,
        dask="allowed",
    )
    
    # Modify the run length
    modified_run_length = (actual_run_length.max() + 1) - actual_run_length
    modified_run_length = modified_run_length.where(~no_data_mask)
    rf_run_length.append(modified_run_length.assign_coords(dekad=interest_period).expand_dims({"dekad": 1}))

da_rf_actual_run_length = xr.concat(rf_run_length, dim="dekad")
da_rf_actual_run_length

CPU times: user 6min 2s, sys: 1.39 s, total: 6min 3s
Wall time: 6min


<xarray.DataArray 'rainfall' (dekad: 372, y: 291, x: 290)>
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
...
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]])
Coordinates:
    spatial_ref  int32 6933
  * y            (y) float64 1.882e+06 1.878e+06 ... 4.375e+05 4.325e+05
  * x            (x) float64 3.182e+06 3.188e+06 ... 4.622e+06 4.628e+06
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-04-21

## Get the long term average of continous deficit in each interest period

In [23]:
# Get the long term average rainfall run length for each interest period 
long_term_avg_run_lenth_rf = {}
for dekad_no_in_year, interest_periods_list in grouped_by_year.items():
    long_term_avg_for_period = da_rf_actual_run_length.sel(dekad=interest_periods_list).mean(dim="dekad")
    long_term_avg_run_lenth_rf[dekad_no_in_year] = long_term_avg_for_period
    
da_rf_long_term_avg_run_length_for_ip = xr.concat([long_term_avg_run_lenth_rf[get_dekad_no_in_year(i)].assign_coords(dekad=i).expand_dims({"dekad": 1}) for  i in interest_periods], dim="dekad")
da_rf_long_term_avg_run_length_for_ip

<xarray.DataArray 'rainfall' (dekad: 372, y: 291, x: 290)>
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
...
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]])
Coordinates:
    spatial_ref  int32 6933
  * y            (y) float64 1.882e+06 1.878e+06 ... 4.375e+05 4.325e+05
  * x            (x) float64 3.182e+06 3.188e+06 ... 4.622e+06 4.628e+06
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-04-21

## Calculate the scaled Precipitation Drought Index

In [24]:
PDI = (da_rf_actual_avg_for_ip / da_rf_long_term_avg_for_ip) * np.sqrt((da_rf_actual_run_length / da_rf_long_term_avg_run_length_for_ip))
PDI

<xarray.DataArray 'rainfall' (dekad: 372, y: 291, x: 290)>
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
...
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]])
Coordinates:
  * y            (y) float64 1.882e+06 1.878e+06 ... 4.375e+05 4.325e+05
  * x            (x) float64 3.182e+06 3.188e+06 ... 4.622e+06 4.628e+06
    spatial_ref  int32 6933
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-04-21

In [25]:
PDI_scaled = (PDI - PDI.min()) / (PDI.max() - PDI.min())
PDI_scaled

<xarray.DataArray 'rainfall' (dekad: 372, y: 291, x: 290)>
array([[[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
...
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]],

       [[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]]])
Coordinates:
  * y            (y) float64 1.882e+06 1.878e+06 ... 4.375e+05 4.325e+05
  * x            (x) float64 3.182e+06 3.188e+06 ... 4.622e+06 4.628e+06
    spatial_ref  int32 6933
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-04-21

In [26]:
# Convert to DataFrame.
df = PDI_scaled.to_dataframe().drop(columns="spatial_ref")
df

rainfall
dekad      y         x                  
2014-01-01 1882500.0 3182500.0       NaN
                     3187500.0       NaN
                     3192500.0       NaN
                     3197500.0       NaN
                     3202500.0       NaN
...                                  ...
2024-04-21 432500.0  4607500.0       NaN
                     4612500.0       NaN
                     4617500.0       NaN
                     4622500.0       NaN
                     4627500.0       NaN

[31393080 rows x 1 columns]

In [27]:
# Convert the DataFrame to an Arrow table
table = Table.from_pandas(df)

# Get the tables existing metadata
existing_meta = table.schema.metadata

# Dump the crs of the DataArray as new metadata to JSON.
meta_json = json.dumps(dict(crs=str(PDI_scaled.rio.crs)))

# Combine the metadata
combined_meta = {
b"xarray.metadata": meta_json.encode(),
**existing_meta,
}

# Replace the metadata.
table = table.replace_schema_metadata(combined_meta)

In [28]:
# Write the table
write_table(table, output_path, compression="GZIP")

---

## Additional information

<b> License </b> The code in this notebook is licensed under the [Apache License, Version 2.0](https://www.apache.org/licenses/LICENSE-2.0).

Digital Earth Africa data is licensed under the [Creative Commons by Attribution 4.0](https://creativecommons.org/licenses/by/4.0/) license.

<b> Contact </b> If you need assistance, please post a question on the [DE Africa Slack channel](https://digitalearthafrica.slack.com/) or on the [GIS Stack Exchange](https://gis.stackexchange.com/questions/ask?tags=open-data-cube) using the `open-data-cube` tag (you can view previously asked questions [here](https://gis.stackexchange.com/questions/tagged/open-data-cube)).

If you would like to report an issue with this notebook, you can file one on [Github](https://github.com/digitalearthafrica/deafrica-sandbox-notebooks).

<b> Compatible datacube version </b>

In [2]:
print(datacube.__version__)

NameError: name 'datacube' is not defined

**Last Tested:**

In [1]:
from datetime import datetime
datetime.today().strftime('%Y-%m-%d')

'2024-11-13'